# Python 中的泛型（Generics）详解

Python 中的泛型（Generics）是类型系统的一部分，虽然它们在日常脚本编写中可能不常见，但在大型项目、库开发或需要严格类型检查的场景中非常有用。

1. 什么是泛型？
泛型`（Generics）`允许你定义参数化的类型，即一个类型可以接受其他类型作为参数。例如：

- `List[int]` 表示“元素类型为 int 的列表”。
- `Dict[str, float]` 表示“键为 str、值为 float 的字典”。
- 泛型的核心目的是提高代码的类型安全性和可读性，同时支持灵活的静态类型检查。

2. Python 泛型的基础
Python 的泛型是通过 typing 模块实现的（Python 3.5+），主要分为两类：

- 容器泛型（如 `List`、`Dict`、`Set`）。
- 自定义泛型（通过 `TypeVar` 或继承 `Generic`）。


## 示例 1：内置容器泛型

In [ ]:
from typing import List, Dict

def process_numbers(numbers: List[int]) -> float:
    return sum(numbers) / len(numbers)

data: List[int] = [1, 2, 3]
result = process_numbers(data)  # 类型检查通过


## 示例 2：类型变量（TypeVar）

In [ ]:
from typing import TypeVar, Sequence

T = TypeVar('T')  # 声明一个泛型类型变量

def first_element(items: Sequence[T]) -> T:
    return items[0]

# 调用时，T 会自动推断为实际类型
print(first_element([1, 2, 3]))    # T 是 int
print(first_element(["a", "b"]))   # T 是 str


## 为什么需要泛型？
场景 1：避免重复代码

假设你要写一个函数，既能处理 int 列表也能处理 str 列表：

In [ ]:
from typing import Sequence, TypeVar

T = TypeVar('T')

def process(items: Sequence[T]) -> T:
    return items[0]

# 无需为每种类型写单独的函数
process([1, 2, 3])     # 返回 int
process(["a", "b"])    # 返回 str


场景 2：明确容器内容的类型

In [ ]:
from typing import Dict

# 明确表示键是 str，值是 int
scores: Dict[str, int] = {"Alice": 90, "Bob": 85}

def update_score(db: Dict[str, int], name: str, value: int) -> None:
    db[name] = value

update_score(scores, "Alice", 95)  # 类型检查通过
update_score(scores, 123, 95)      # 类型检查会报错（键必须是 str）


## 4. 自定义泛型类
通过继承 Generic，可以创建自己的泛型类：

In [ ]:
from typing import Generic, TypeVar, List

T = TypeVar('T')

class Stack(Generic[T]):
    def __init__(self) -> None:
        self.items: List[T] = []

    def push(self, item: T) -> None:
        self.items.append(item)

    def pop(self) -> T:
        return self.items.pop()

# 使用示例
int_stack = Stack[int]()
int_stack.push(1)
int_stack.push("hello")  # 类型检查会报错（应为 int）

str_stack = Stack[str]()
str_stack.push("world")


## 5. 泛型的约束
可以用 bound 限制泛型类型的范围：

In [ ]:
from typing import TypeVar, Number

N = TypeVar('N', bound=Number)  # 只能是 Number 的子类（如 int, float）

def add(a: N, b: N) -> N:
    return a + b

add(1, 2)      # 正确
add(1.5, 3.2)   # 正确
add("a", "b")   # 类型检查报错


## 6. 动态类型 vs 静态类型中的泛型
- 动态类型：Python 运行时不会强制泛型约束（List[int] 和 List[str] 运行时是同一个类）。
- 静态类型检查：工具如 mypy 会根据泛型类型进行验证。

In [ ]:
from typing import List

x: List[int] = [1, 2, 3]
x.append("hello")  # 运行时不会报错，但 mypy 会标记为错误


## 7. 实际应用场景
API 设计：明确函数参数和返回值的类型关系。

In [ ]:
def parse_response(response: Dict[str, Any]) -> Optional[User]:
    ...


库开发：如 pandas 的 DataFrame、numpy 的 ndarray 可以用泛型标注。

团队协作：大型项目中减少类型相关的 bug。

## 协变与逆变
### 协变(covariant)
定义：如果类 A 是类 B 的子类，则类 Container[A] 是类 Container[B] 的子类，这样的关系称为协变。
关键点：协变通常用于输出类型。当一个泛型容器或函数返回子类型的对象时，可以使用协变。
协变示例

假设我们有一个动物类和一个狗类，狗类继承自动物类。我们还有一个返回动物的函数，但如果我们知道这个函数实际上总是返回狗，协变就允许我们将这个函数的返回类型声明为狗。

In [ ]:
from typing import List, Generic, TypeVar

T_co = TypeVar('T_co', covariant=True)

class Animal:
    def make_sound(self):
        print("Some sound")

class Dog(Animal):
    def make_sound(self):
        print("Bark")

class Cat(Animal):
    def make_sound(self):
        print("Meow")

# 创建一个泛型容器类，它是协变的
class AnimalShelter(Generic[T_co]):
    def __init__(self, animal: T_co):
        self.animal = animal
    
    def get_animal(self) -> T_co:
        return self.animal


animal_shelter = AnimalShelter(Animal)
animal: Animal = AnimalShelter.get_animal() 
animal.make_sound() 

# 创建一个 Dog/Cat Shelter 子类容器
dog_shelter = AnimalShelter(Dog())
cat_shelter = AnimalShelter(Cat())
#  这些也是Animal的子类
dog: Animal = dog_shelter.get_animal() 
cat: Animal = cat_shelter.get_animal() 
dog.make_sound()  # 输出: Bark
cat.make_sound()  # 输出: Meow

AnimalShelter 类定义为协变，意味着：

AnimalShelter[Dog] 被视为 AnimalShelter[Animal] 的子类型。

这在类型系统中是安全的，因为它只从容器中输出动物（通过 get_animal 方法）。

### 逆变(contravariant)
定义：如果类 A 是类 B 的子类，则类 Container[B] 是类 Container[A] 的子类，这样的关系称为逆变。
关键点：逆变通常用于输入类型。允许一个接受更特定类型参数（如Dog）的函数被视为接受更一般类型（如Animal）的函数。在设计如事件处理系统或回调机制时特别有用，你可以根据需要传入特定类型的处理函数，而不破坏系统的整体类型安全性。
逆变示例

假设我们有一个需要动物作为输入的函数，如果这个函数可以接受任何动物，那么使用逆变可以让我们传入任何特定动物的父类。

In [ ]:
from typing import TypeVar, Generic

# 定义一个逆变的类型变量
T_contra = TypeVar('T_contra', contravariant=True)

class Animal:
    def eat(self):
        print("This animal is eating.")

class Cat(Animal):
    def eat(self):
        print("Cat is eating.")

# 定义泛型 Feeder 类
class Feeder(Generic[T_contra]):
    def feed(self, animal: T_contra):
        print(f"Feeding a {animal.__class__.__name__}:")
        animal.eat()
        
# 定义一个函数，它接受一个能够喂食 Cat 的 Feeder 对象
def setup_feeder(feeder: Feeder[Cat]):
    feeder.feed(Cat())  

# 实例化一个能够喂食任何 Animal 的 Feeder
animal_feeder = Feeder[Animal]()

# 尝试使用 AnimalFeeder 喂食 Cat，由于逆变而有效
setup_feeder(animal_feeder)


在这个例子中，Feeder 类型参数 T_contra 是逆变的，意味着：Feeder[Animal]会被视为Feeder[Cat]的子类。因此我们可以在期望 Feeder[Cat] 的地方使用 Feeder[Animal]。setup_feeder函数期望得到Feeder[Cat]类型，由于逆变，我们可以传递一个 animal_feeder 实例。

## 匹配工具


### 静态类型检查工具
静态类型检查是在程序运行前进行的，可以帮助开发者在代码执行之前发现潜在的类型错误。

- mypy

简介: mypy 是一个流行的Python静态类型检查工具，它可以帮助识别类型不匹配的问题。

安装: 可以通过pip安装mypy。
`pip install mypy`

基本使用: 在命令行中运行mypy检查一个文件。

`mypy example.py`
配置文件: 可以在项目根目录下创建一个 mypy.ini 文件来定制mypy的行为。


```shell
[mypy]
check_untyped_defs = True
ignore_missing_imports = True
```

### 运行时类型检查
运行时类型检查是在程序执行时进行的，可以捕获那些静态类型检查可能漏掉的类型错误。

- pydantic库

简介: pydantic 是一个使用Python类型注释进行数据验证和设置管理的库，它在运行时强制类型安全。

安装: 通过pip安装pydantic。

`pip install pydantic`

基本使用: 使用 pydantic 创建数据模型。

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Alice", age="thirty")  # Error: value is not a valid integer


在上例中，尝试将字符串 "thirty" 用作整数会引发错误。

#### Field函数
Field 是 pydantic 提供的一个函数，用于为模型字段定义更详细的验证和元数据。通过使用 Field，你可以指定字段的默认值、别名、验证规则等。

基本用法

定义默认值和验证条件： Field 可以用来定义字段的默认值和一系列验证条件，例如字段的最大长度、范围、正则表达式等。
别名设置： Field 还允许你为模型的字段设置序列化和反序列化时使用的别名。
示例：